# Syntropy2 — round 2: `mid` tier on the mixed-format corpus

Trains the **mid** model (d=320, 5 layers, ~15.7M params) on `corpus_r2` (code, docs/markup, structured text, prose, Wikipedia) for the nsc codec's neural text path.

**Setup:** accelerator **GPU T4**, and attach the dataset **`christianocanab/syntropy2-r2`** (it carries the trainer script, the corpus, and — after the first session — the resume state under `runs/`).

**Sessions:** three sessions of ~10 h (30 h budget). Session 1: `FRESH_START = True`. After each session, download the notebook output `runs/` folder, push it with `tools/push_r2_session.sh <runs dir>`, then run the next session with `FRESH_START = False`.

This notebook imports the trainer from the dataset instead of carrying a copy of the code, so what runs here is exactly the tested `neuro_symbolic_compressor.py`.


In [ ]:
# =============================================================================
#  BOOTSTRAP -- import the trainer from the attached dataset
# =============================================================================
import os, sys, glob, subprocess, importlib

try:
    import tokenizers  # must be importable BEFORE the trainer module is imported
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.15"])

_mods = sorted(glob.glob("/kaggle/input/**/neuro_symbolic_compressor.py", recursive=True)) \
        or sorted(glob.glob("./neuro_symbolic_compressor.py"))
if not _mods:
    raise RuntimeError("neuro_symbolic_compressor.py not found — attach the dataset "
                       "christianocanab/syntropy2-r2 to this notebook.")
sys.path.insert(0, os.path.dirname(_mods[0]))
import neuro_symbolic_compressor as nsc
importlib.reload(nsc)
nsc.ensure_deps()

import torch
assert nsc._HAS_TOKENIZERS, "tokenizers is not available — the BPE tokenizer cannot be trained"
print("trainer :", _mods[0])
print("device  :", nsc.DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY")
print("corpus  :", nsc.find_corpus("corpus_r2"))
if not torch.cuda.is_available():
    print("WARNING: no GPU — enable the T4 accelerator before running the RUN cell.")


In [ ]:
# =============================================================================
#  RUN CELL  (only FRESH_START changes between sessions)
# =============================================================================
TOTAL_BUDGET_H = 30      # sizes the curriculum — keep constant across sessions
SESSION_CAP_H  = 9.8     # wall-clock hours for THIS session (3 sessions = one weekly T4 quota)
FRESH_START    = False   # True only for the very first session; False to resume
TARGET_PHASES  = 2       # train/benchmark slots per session
TIER, DATASET  = "mid", "corpus_r2"

import shutil
OUTDIR = "/kaggle/working/runs" if os.path.isdir("/kaggle/working") else os.path.join(nsc.WORKDIR, "runs")

# Previous session's state: the folder holding ledger.json inside the attached dataset.
_prev = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ledger.json", recursive=True))
PREV_RUNS = _prev[0] if _prev else None

if FRESH_START and PREV_RUNS:
    raise RuntimeError(f"FRESH_START=True but a previous run exists at {PREV_RUNS}. "
                       "Set FRESH_START=False to resume (True would discard that training).")
if not FRESH_START:
    if not PREV_RUNS:
        raise RuntimeError("resume expected but no ledger.json under /kaggle/input — push the "
                           "previous session's runs/ to the dataset (tools/push_r2_session.sh) "
                           "and re-attach it, or set FRESH_START=True for the first session.")
    print(f"[resume] copying {PREV_RUNS} -> {OUTDIR}")
    shutil.copytree(PREV_RUNS, OUTDIR, dirs_exist_ok=True)

result = nsc.run_budget(
    total_budget_h=TOTAL_BUDGET_H,
    session_cap_h=SESSION_CAP_H,
    outdir=OUTDIR,
    fresh_start=FRESH_START,
    target_phases=TARGET_PHASES,
    goal_test_bpb=None,          # this round stops on curriculum end or a val plateau
    tier=TIER, dataset=DATASET,
)

best = result["best"]
print("\nBEST:", "none" if best is None
      else f"phase {best['phase']}  calibrated test bpb={best['test_bpb_cal']:.4f}")
for row in result["ledger"]:
    if "domains_cal" in row:
        print(f"  phase {row['phase']} step {row['step']:,}/{row['total_steps']:,}: "
              + "  ".join(f"{k}={v:.4f}" for k, v in row["domains_cal"].items()))
print(f"Accumulated: {result['accumulated_h']:.2f}h")
stopped = result.get("stopped") or {}
if stopped.get("done"):
    print("\nTRAINING FINISHED —", stopped["reason"])
    print("Download runs/ and install the model with tools/install_model.sh.")
else:
    print("\nNot finished: download runs/, push it (tools/push_r2_session.sh), "
          "then run again with FRESH_START=False.")
